# CP1 · B5 — Home/Office: so bán kính stay và rule

Chọn kernel **Python (GeoLife)** rồi **Run All**. Notebook độc lập, không cần chạy lại NB02–09.
Các cờ đã bật sẵn; chạy full-data khi bạn bấm chạy notebook. Assistant mới kiểm thử trên dữ liệu giả.

- Hai nhánh **fixed-anchor 100 m/10 phút** và **150 m/10 phút**, tái dùng stays B4.
- Hai cách gán vai trò: **score hiện tại** và **rule đơn giản**.
- Báo riêng cho gom địa điểm **heuristic-v1** và **dbscan-v1**, đủ **182 user** kể cả unknown.
- Phần context đọc **24.177.772 GPS fixes trong cache B4** theo trajectory để tính coverage và timezone;
  không chạy lại detector hay preprocessing. Một tiến trình, checkpoint theo user.
- Giữ baseline/B4 nguyên trạng. Lựa chọn 150 m là quyết định thử nghiệm downstream của người dùng,
  không phải tuyên bố đã thắng cổng B4. Không đo accuracy Home/Office khi chưa có đáp án thật.

Protocol: `reports/cp1/role_protocol.md`. Đầu ra: `artifacts/benchmark/b5/roles_v1`.


## 1. Thiết lập

Giữ các cờ `True` để chạy toàn bộ B5. `RESUME_B5=True` tái dùng user completed khi checksum khớp.
Interrupt sẽ dừng job trong kernel, giữ checkpoint đã xong; Run All tiếp tục phần còn thiếu.
Không sửa module/config/protocol sau khi đã khóa; cần phiên bản đầu ra mới nếu đổi phương pháp.


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/geolife_inference').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from geolife_inference import benchmark_roles as b5
b5 = importlib.reload(b5)
OUT_B5 = Path(globals().get('OUT_B5', ROOT / 'artifacts/benchmark/b5/roles_v1'))
RESUME_B5 = globals().get('RESUME_B5', True)
def completed_b5(stage):
    path = OUT_B5 / stage / 'audit.json'
    return path.exists() and b5._read(path)['status'] == 'completed'

assert Path(sys.prefix).resolve() == (ROOT / '.venv').resolve(), 'Chọn kernel Python (GeoLife).'
RUN_B5_MAIN = True
RUN_B5_CONTEXT = True
RUN_B5_ABLATIONS = True
RUN_B5_SUMMARY = True
RUN_B5_PLOTS = True
RESUME_B5 = True
print('Đầu ra B5:', OUT_B5)
print('Các bước: khóa nguồn → so rule/radius → coverage/timezone → ablations → bảng và plot')


## 2. Khóa protocol và nguồn

**Bán kính stay** là 100/150 m; **eps gom địa điểm** vẫn 100 m trong cả hai nhánh.
Mỗi nhánh được gom địa điểm riêng, không chép cluster IDs từ nhánh kia.

Score hiện tại: recurrence 30%, duration 25%, temporal consistency 20%, coverage 15%, relative strength 10%.
Giữ nguyên các ngưỡng và điều kiện unknown trong NB05.

Rule đơn giản: cùng điều kiện đủ ngày/dwell/tuần; Home ưu tiên số đêm ghé rồi dwell đêm,
Office ưu tiên số ngày ghé giờ làm việc rồi dwell tương ứng. Hòa cả hai thì unknown.
Rule đơn giản không có cổng score/share/margin; nhiều prediction hơn chưa có nghĩa tốt hơn.


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/geolife_inference').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from geolife_inference import benchmark_roles as b5
b5 = importlib.reload(b5)
OUT_B5 = Path(globals().get('OUT_B5', ROOT / 'artifacts/benchmark/b5/roles_v1'))
RESUME_B5 = globals().get('RESUME_B5', True)
def completed_b5(stage):
    path = OUT_B5 / stage / 'audit.json'
    return path.exists() and b5._read(path)['status'] == 'completed'

lock_b5 = b5.register(ROOT, OUT_B5)
design = pd.DataFrame([{'stay': f'{r} m / 10 phút', 'grouping': m, 'rule': rule}
                       for r in b5.RADII for m in b5.MODEL_VERSIONS for rule in b5.RULES])
display(design)
print('User theo manifest:', len(lock_b5['users']))
used_b5 = b5._used(ROOT, OUT_B5)
print(f'Compute tích lũy: {used_b5/3600:.3f}/24 giờ; còn {(86400-used_b5)/3600:.3f} giờ')


## 3. So 100/150 m và hai rule trên toàn dataset

Mỗi user: đọc hai bộ stay đã có, gom địa điểm theo hai phương pháp, tính features và chọn Home/Office.
Nhánh **100 m + score hiện tại** phải khớp kết quả NB05 trước khi tiếp tục.

Mỗi rule/nhánh có đúng một hàng Home và một hàng Office cho mọi user; không loại người thiếu dữ liệu.
Đọc tỷ lệ unknown cùng reason và coverage; không tối đa hóa prediction rate.


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/geolife_inference').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from geolife_inference import benchmark_roles as b5
b5 = importlib.reload(b5)
OUT_B5 = Path(globals().get('OUT_B5', ROOT / 'artifacts/benchmark/b5/roles_v1'))
RESUME_B5 = globals().get('RESUME_B5', True)
def completed_b5(stage):
    path = OUT_B5 / stage / 'audit.json'
    return path.exists() and b5._read(path)['status'] == 'completed'

b5.register(ROOT, OUT_B5)
if globals().get('RUN_B5_MAIN', True):
    display(b5.run_main(ROOT, OUT_B5, resume=RESUME_B5))
    b5.summarize(ROOT, OUT_B5, include_ablations=False, resume=RESUME_B5)
    display(pd.read_csv(OUT_B5 / 'summary_main/rates.csv'))
    display(pd.read_csv(OUT_B5 / 'summary_main/comparisons.csv'))
else:
    print('Phần so chính đang tắt.')


## 4. Coverage, timezone và hai khối thời gian

Đọc cache GPS B4 **một lượt**, theo từng trajectory. Có log user đang xử lý và user đã checkpoint.
Tính lại coverage cho cửa sổ ±1 giờ; timezone theo từng fix; hai khối 28 ngày đầu tiên của từng user.
Coverage UTC cơ sở phải khớp NB05, tổng points/trajectories phải khớp manifest.

Timezone được tra bằng thư viện đã cài trong môi trường notebook; offset lịch sử dùng IANA/zoneinfo.
Không gán một zone đa số cho người đi nhiều nơi. Fix không xác định zone dùng fallback và báo riêng.
Pair đi qua biên zone gán theo fix trái (tối đa 5 phút), ghi số lượng/thời lượng để biết giới hạn.


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/geolife_inference').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from geolife_inference import benchmark_roles as b5
b5 = importlib.reload(b5)
OUT_B5 = Path(globals().get('OUT_B5', ROOT / 'artifacts/benchmark/b5/roles_v1'))
RESUME_B5 = globals().get('RESUME_B5', True)
def completed_b5(stage):
    path = OUT_B5 / stage / 'audit.json'
    return path.exists() and b5._read(path)['status'] == 'completed'

b5.register(ROOT, OUT_B5)
if globals().get('RUN_B5_CONTEXT', True):
    context_audit_b5 = b5.run_context(ROOT, OUT_B5, resume=RESUME_B5)
    print('GPS fixes:', context_audit_b5['processed_points'])
    print('Trajectories:', context_audit_b5['processed_trajectories'])
    print('User có zone khác/fallback:', context_audit_b5['affected_users'])
    print('Fix dùng fallback:', context_audit_b5['unknown_zone_points'])
else:
    print('Phần context đang tắt; các ablation cần context completed.')


## 5. Chạy các kiểm tra độ nhạy

- **Giờ −1/+1:** tính lại coverage/features, giữ các ngưỡng và grouping.
- **Timezone:** giữ grouping; tính features cluster theo zone tại tâm và coverage theo GPS local time.
- **Hai khối 28 ngày:** chỉ dùng stays nằm trọn trong từng khối và gom địa điểm riêng;
  ghi số stay giao biên bị loại. Không dùng clusters/context của tương lai.

Để so theo thời gian, mỗi user/role cần cả hai khối có ≥7 ngày GPS và ≥60 phút coverage trong khung role.
Nếu dưới 15 user/role thì báo chưa đủ cohort, không gọi là đã đánh giá ổn định.


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/geolife_inference').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from geolife_inference import benchmark_roles as b5
b5 = importlib.reload(b5)
OUT_B5 = Path(globals().get('OUT_B5', ROOT / 'artifacts/benchmark/b5/roles_v1'))
RESUME_B5 = globals().get('RESUME_B5', True)
def completed_b5(stage):
    path = OUT_B5 / stage / 'audit.json'
    return path.exists() and b5._read(path)['status'] == 'completed'

b5.register(ROOT, OUT_B5)
if globals().get('RUN_B5_ABLATIONS', True):
    if completed_b5('main') and completed_b5('context'):
        display(b5.run_ablations(ROOT, OUT_B5, resume=RESUME_B5))
    else:
        print('Cần main và context completed trước. Chạy mục 3–4, không chạy lại detector B4.')
else:
    print('Các ablation đang tắt.')


## 6. Đọc bảng và plot — hiển thị ảnh ngay dưới cell

- **Tỷ lệ dự đoán:** mọi user đều vào mẫu số, phần còn lại là unknown.
- **Agreement địa điểm:** chỉ tính trên user cả hai phía đều predicted, cách tâm ≤200 m.
  Không dùng cluster ID để kết luận cùng nơi. Luôn đọc cột `both_predicted`.
- **Transitions:** bao nhiêu predicted→unknown và unknown→predicted.
- **Cờ điều tra:** >10% user cùng predicted đổi nơi >200 m, hoặc tỷ lệ predicted đổi >5 điểm %.
- Timezone báo cả toàn dataset và riêng user bị ảnh hưởng; không để phần không đổi che lấp tác động.

Không gọi agreement là accuracy. Nơi ở/công việc thật có thể đổi qua thời gian.


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/geolife_inference').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from geolife_inference import benchmark_roles as b5
b5 = importlib.reload(b5)
OUT_B5 = Path(globals().get('OUT_B5', ROOT / 'artifacts/benchmark/b5/roles_v1'))
RESUME_B5 = globals().get('RESUME_B5', True)
def completed_b5(stage):
    path = OUT_B5 / stage / 'audit.json'
    return path.exists() and b5._read(path)['status'] == 'completed'

b5.register(ROOT, OUT_B5)
if globals().get('RUN_B5_SUMMARY', True) and completed_b5('main'):
    include_ablations_b5 = completed_b5('ablations')
    b5.summarize(ROOT, OUT_B5, include_ablations=include_ablations_b5, resume=RESUME_B5)
    summary_dir_b5 = OUT_B5 / ('summary' if include_ablations_b5 else 'summary_main')
    display(pd.read_csv(summary_dir_b5 / 'rates.csv'))
    comparisons_b5 = pd.read_csv(summary_dir_b5 / 'comparisons.csv')
    display(comparisons_b5)
    display(pd.read_csv(summary_dir_b5 / 'reasons.csv'))
    display(pd.read_csv(summary_dir_b5 / 'coverage_strata.csv'))
    if include_ablations_b5:
        display(pd.read_csv(summary_dir_b5 / 'timezone_inventory.csv'))
        display(comparisons_b5[comparisons_b5.comparison.eq('temporal')])
    else:
        print('Đang hiển thị so chính; context/ablations chưa completed nên B5 chưa đủ toàn bộ phạm vi.')
    if globals().get('RUN_B5_PLOTS', True):
        b5.render(ROOT, OUT_B5, include_ablations=include_ablations_b5, resume=RESUME_B5)
        plot_dir_b5 = OUT_B5 / ('plots' if include_ablations_b5 else 'plots_main')
        for plot_name_b5 in b5._read(plot_dir_b5 / 'audit.json')['output_hashes']:
            print(plot_name_b5)
            display(Image(filename=str(plot_dir_b5 / plot_name_b5)))
    used_b5 = b5._used(ROOT, OUT_B5)
    print(f'Compute tích lũy: {used_b5/3600:.3f}/24 giờ; còn {(86400-used_b5)/3600:.3f} giờ')
else:
    print('Cần main completed và RUN_B5_SUMMARY=True để dựng bảng/plot.')


## 7. Chốt sau khi đã đọc kết quả

B5 đủ khi main/context/ablations/summary/plots completed đúng scope và checksum, hoặc có phần thiếu
được ghi rõ với nguyên nhân. Code chạy xong không tự chứng minh rule đúng hoặc tự đổi phiên bản service.

Cần trả lời: 150 m có thay Home/Office so với 100 m không; hai rule chọn cùng nơi bao nhiêu;
unknown xuất phát từ thiếu dữ liệu hay điều kiện rule; dịch giờ/timezone có đổi đáng kể không;
so thời gian đủ cohort chưa. Chưa có kết quả chạy thì chưa tick PLAN.
